In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
print(bool(os.getenv("ANTHROPIC_API_KEY")))

True


In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


In [4]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

In [5]:
def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    for attempt in range(3):
        message = client.messages.create(**params)

        text_blocks = []

        for block in message.content:
            text = getattr(block, "text", None)
            if text:
                text_blocks.append(text)

        if text_blocks:
            return "\n".join(text_blocks)

    return None

In [6]:
evaluation_dataset = [
    {
        "name": "Athlete maintaining weight while improving power",
        "prompt_inputs": {
            "height": "180 cm",
            "weight": "75 kg",
            "goal": "Maintain weight and improve power",
            "restrictions": "Lactose intolerance"
        },
        "solution_criteria": [
            "Provides a one-day meal plan",
            "Includes an estimated daily calorie amount",
            "Includes protein, fat, and carbohydrate amounts",
            "Specifies meal timing",
            "Uses foods compatible with lactose intolerance",
            "Includes portion sizes"
        ]
    },
    {
        "name": "Athlete seeking a heart-conscious meal plan",
        "prompt_inputs": {
            "height": "170 cm",
            "weight": "70 kg",
            "goal": "Maintain fitness and improve cholesterol levels",
            "restrictions": "High cholesterol"
        },
        "solution_criteria": [
            "Provides a one-day meal plan",
            "Uses foods appropriate for high cholesterol",
            "Includes an estimated calorie target",
            "Includes macronutrient amounts",
            "Includes meal timing",
            "Includes portion sizes"
        ]
    },
    {
        "name": "Athlete needing post-race recovery nutrition",
        "prompt_inputs": {
            "height": "165 cm",
            "weight": "55 kg",
            "goal": "Post-race recovery",
            "restrictions": "None"
        },
        "solution_criteria": [
            "Provides a one-day recovery meal plan",
            "Includes adequate carbohydrates for recovery",
            "Includes protein for recovery",
            "Includes an estimated calorie amount",
            "Includes meal timing",
            "Includes portion sizes"
        ]
    }
]

In [7]:
def run_prompt(prompt_inputs):
    prompt = f"""
Create a complete one-day meal plan for an athlete with the following profile:

Height: {prompt_inputs["height"]}
Weight: {prompt_inputs["weight"]}
Goal: {prompt_inputs["goal"]}
Dietary restriction or health consideration: {prompt_inputs["restrictions"]}

Requirements:
1. Provide meals and snacks for one full day.
2. Include an estimated daily calorie target.
3. Include estimated protein, carbohydrate, and fat amounts.
4. Give specific meal timing.
5. Give specific portion sizes.
6. Respect the athlete's dietary restriction or health consideration.
7. Tailor the plan to the athlete's stated goal.
8. Keep the response practical, organized, and easy to follow.
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [8]:
test_output = run_prompt(evaluation_dataset[0]["prompt_inputs"])
print(test_output)

# One-Day Meal Plan: Power Athlete (Lactose Intolerant)

## Daily Targets
| Metric | Target |
|---|---|
| **Calories** | ~3,100–3,150 kcal |
| **Protein** | ~215 g (2.9 g/kg) |
| **Carbohydrates** | ~320 g (4.3 g/kg) |
| **Fat** | ~105–110 g (30%) |

*Rationale: Calorie level supports weight maintenance with high training demand. Higher protein supports muscle power/recovery; carbs are timed around training to fuel explosive output; all dairy replaced with lactose-free or naturally low-lactose alternatives.*

---

## 6:30 AM — Breakfast (Pre-Training Fuel)
- Oats, dry — 80 g, cooked with **lactose-free milk** — 250 ml
- Banana — 1 medium
- Peanut butter — 1 tbsp (16 g)
- Cinnamon to taste

**Macros:** ~600 kcal | 23 g protein | 90 g carbs | 17 g fat

---

## 9:00–10:30 AM — Training Session
*(Strength/power work — explosive lifts, plyometrics, sprint work)*

---

## 10:30 AM — Post-Training Recovery Shake
- Whey protein **isolate** (minimal/no lactose) — 1 scoop (30 g)
- Banana — 1 med

In [9]:
import json

def grade_output(test_case, output):
    criteria_text = "\n".join(
        f"- {criterion}" for criterion in test_case["solution_criteria"]
    )

    grading_prompt = f"""
You are evaluating an AI-generated athlete meal plan.

Athlete profile:
{json.dumps(test_case["prompt_inputs"], indent=2)}

Solution criteria:
{criteria_text}

AI response:
<response>
{output}
</response>

Evaluate the response against the solution criteria.

Return valid JSON only in exactly this format:
{{
    "score": 8,
    "reasoning": "Brief explanation of why the response earned this score."
}}

Use a score from 1 to 10.
Do not use markdown code fences.
"""

    messages = []
    add_user_message(messages, grading_prompt)

    result = chat(messages)

    if result is None:
        return {
            "score": 1,
            "reasoning": "The grader did not return a usable response."
        }

    cleaned = (
        result.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )

    return json.loads(cleaned)

In [10]:
test_grade = grade_output(
    evaluation_dataset[0],
    test_output
)

print(test_grade)

{'score': 9, 'reasoning': "The response fully satisfies all solution criteria: it provides a complete one-day meal plan with specific times (6:30 AM, 10:30 AM, 12:30 PM, 3:30 PM, 6:30-7:00 PM, 9:00 PM), a clear total daily calorie estimate (~3,100-3,150 kcal), and detailed macro breakdowns (protein ~215g, carbs ~320g, fat ~105-110g) both at the daily level and per meal. Portion sizes are specific and practical (g, cups, tbsp). Lactose intolerance is carefully addressed throughout—using lactose-free milk, lactose-free yogurt, lactose-free cottage cheese, and flagging potential lactose traces in whey isolate with a substitution suggestion. The macro math across meals sums closely to the stated daily totals, showing internal consistency. Minor issues include a slight protein overestimate for the lunch chicken breast portion and a truncated/missing unit label in the final snack's fat value, but these are small and do not undermine the overall quality or usability of the plan."}


In [11]:
evaluation_results = []

for test_case in evaluation_dataset:
    output = None

    while output is None:
        output = run_prompt(test_case["prompt_inputs"])

    grade = grade_output(test_case, output)

    evaluation_results.append({
        "name": test_case["name"],
        "prompt_inputs": test_case["prompt_inputs"],
        "solution_criteria": test_case["solution_criteria"],
        "output": output,
        "score": grade["score"],
        "reasoning": grade["reasoning"]
    })

print(json.dumps(evaluation_results, indent=2))

[
  {
    "name": "Athlete maintaining weight while improving power",
    "prompt_inputs": {
      "height": "180 cm",
      "weight": "75 kg",
      "goal": "Maintain weight and improve power",
      "restrictions": "Lactose intolerance"
    },
    "solution_criteria": [
      "Provides a one-day meal plan",
      "Includes an estimated daily calorie amount",
      "Includes protein, fat, and carbohydrate amounts",
      "Specifies meal timing",
      "Uses foods compatible with lactose intolerance",
      "Includes portion sizes"
    ],
    "output": "# One-Day Meal Plan: Athlete Performance & Power\n\n## Athlete Profile Summary\n- **Height/Weight:** 180 cm / 75 kg\n- **Goal:** Weight maintenance + power improvement\n- **Restriction:** Lactose intolerance (dairy-free or lactose-free dairy substitutes used)\n\n## Daily Targets\n\n| Metric | Target |\n|---|---|\n| **Calories** | ~2,900 kcal |\n| **Protein** | 165 g (~2.2 g/kg) \u2014 supports muscle power/recovery |\n| **Carbohydrates*

In [12]:
from statistics import mean

scores = [result["score"] for result in evaluation_results]

average_score = mean(scores)
pass_count = sum(score >= 7 for score in scores)
pass_rate = (pass_count / len(scores)) * 100

print("Prompt Evaluation Summary")
print("-------------------------")
print(f"Test cases: {len(scores)}")
print(f"Average score: {average_score:.1f}/10")
print(f"Pass rate: {pass_rate:.0f}%")
print()

for index, result in enumerate(evaluation_results, start=1):
    status = "PASS" if result["score"] >= 7 else "FAIL"
    print(f"Test case {index}: {result['score']}/10 - {status}")

Prompt Evaluation Summary
-------------------------
Test cases: 3
Average score: 9.3/10
Pass rate: 100%

Test case 1: 9/10 - PASS
Test case 2: 9/10 - PASS
Test case 3: 10/10 - PASS


In [13]:
from html import escape
from pathlib import Path

def generate_html_report(results, filename="prompt_evaluation_report.html"):
    scores = [result["score"] for result in results]
    average_score = mean(scores)
    pass_count = sum(score >= 7 for score in scores)
    pass_rate = (pass_count / len(scores)) * 100

    sections = []

    for result in results:
        status = "PASS" if result["score"] >= 7 else "FAIL"

        criteria_html = "".join(
            f"<li>{escape(item)}</li>"
            for item in result["solution_criteria"]
        )

        sections.append(f"""
        <section class="case">
            <h2>{escape(result["name"])}</h2>

            <p>
                <strong>Score:</strong> {result["score"]}/10
                &nbsp;&nbsp;
                <strong>Status:</strong> {status}
            </p>

            <h3>Prompt Inputs</h3>
            <pre>{escape(str(result["prompt_inputs"]))}</pre>

            <h3>Solution Criteria</h3>
            <ul>
                {criteria_html}
            </ul>

            <h3>Grader Reasoning</h3>
            <p>{escape(result["reasoning"])}</p>

            <h3>Model Output</h3>
            <pre>{escape(result["output"] or "None")}</pre>
        </section>
        """)

    html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Prompt Evaluation Report</title>

    <style>
        body {{
            font-family: Arial, sans-serif;
            max-width: 1000px;
            margin: 40px auto;
            padding: 0 24px;
            line-height: 1.6;
        }}

        .summary {{
            border: 1px solid #cccccc;
            border-radius: 10px;
            padding: 18px;
            margin: 24px 0 32px;
        }}

        .case {{
            border-top: 2px solid #dddddd;
            padding-top: 24px;
            margin-top: 32px;
        }}

        pre {{
            white-space: pre-wrap;
            word-wrap: break-word;
            background: #f6f6f6;
            padding: 14px;
            border-radius: 8px;
        }}
    </style>
</head>

<body>

    <h1>Prompt Evaluation Report</h1>
    <p>Evaluation of athlete meal-plan prompt performance</p>

    <div class="summary">
        <h2>Summary</h2>

        <p><strong>Test Cases:</strong> {len(results)}</p>
        <p><strong>Average Score:</strong> {average_score:.1f}/10</p>
        <p><strong>Pass Rate:</strong> {pass_rate:.0f}%</p>
    </div>

    {''.join(sections)}

</body>
</html>
"""

    Path(filename).write_text(html, encoding="utf-8")

    print(f"Report saved as: {filename}")


generate_html_report(evaluation_results)

Report saved as: prompt_evaluation_report.html
